# UEFA Champions League 2025/26 — SQL + Python Analysis

This notebook reads the SQLite database produced by `src/run_pipeline.py`.

The complete results source covers 189 matches. Detailed possession/shooting statistics cover the 144-match league phase only.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB = Path("../data/processed/champions_league.db")
conn = sqlite3.connect(DB)


## 1. Competition overview

In [ ]:
pd.read_sql_query(
    '''
    SELECT
        stage,
        COUNT(*) AS matches,
        SUM(home_goals + away_goals) AS goals,
        ROUND(1.0 * SUM(home_goals + away_goals) / COUNT(*), 2) AS goals_per_match
    FROM matches
    GROUP BY stage
    ORDER BY CASE stage
        WHEN 'League Phase' THEN 1
        WHEN 'Knockout Play-offs' THEN 2
        WHEN 'Round of 16' THEN 3
        WHEN 'Quarter-finals' THEN 4
        WHEN 'Semi-finals' THEN 5
        WHEN 'Final' THEN 6
    END
    ''',
    conn
)


## 2. Reconstruct the league-phase table

In [ ]:
standings = pd.read_sql_query(
    '''
    WITH team_matches AS (
        SELECT t.team_name,
               m.home_goals AS gf,
               m.away_goals AS ga,
               CASE WHEN m.home_goals > m.away_goals THEN 3
                    WHEN m.home_goals = m.away_goals THEN 1 ELSE 0 END AS pts
        FROM matches m
        JOIN teams t ON t.team_id = m.home_team_id
        WHERE m.stage = 'League Phase'

        UNION ALL

        SELECT t.team_name,
               m.away_goals,
               m.home_goals,
               CASE WHEN m.away_goals > m.home_goals THEN 3
                    WHEN m.away_goals = m.home_goals THEN 1 ELSE 0 END
        FROM matches m
        JOIN teams t ON t.team_id = m.away_team_id
        WHERE m.stage = 'League Phase'
    )
    SELECT team_name,
           COUNT(*) AS played,
           SUM(pts) AS points,
           SUM(gf) AS goals_for,
           SUM(ga) AS goals_against,
           SUM(gf) - SUM(ga) AS goal_difference
    FROM team_matches
    GROUP BY team_name
    ORDER BY points DESC, goal_difference DESC, goals_for DESC
    ''',
    conn
)
standings.head(12)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top12 = standings.head(12)
ax.bar(top12["team_name"], top12["points"])
ax.set_title("League Phase — Top 12 by Points")
ax.set_ylabel("Points")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()


## 3. Home advantage

In [ ]:
pd.read_sql_query(
    '''
    SELECT
        CASE match_outcome
            WHEN 'H' THEN 'Home Win'
            WHEN 'D' THEN 'Draw'
            WHEN 'A' THEN 'Away Win'
        END AS outcome,
        COUNT(*) AS matches,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS share_pct
    FROM matches
    WHERE stage = 'League Phase'
    GROUP BY match_outcome
    ORDER BY matches DESC
    ''',
    conn
)


## 4. Attacking efficiency

In [ ]:
efficiency = pd.read_sql_query(
    '''
    WITH team_stats AS (
        SELECT ht.team_name,
               m.home_goals AS goals,
               s.home_shots_total AS shots,
               s.home_shots_on_target_count AS shots_on_target,
               s.home_possession AS possession
        FROM league_phase_stats s
        JOIN matches m ON m.match_id = s.match_id
        JOIN teams ht ON ht.team_id = m.home_team_id

        UNION ALL

        SELECT at.team_name,
               m.away_goals,
               s.away_shots_total,
               s.away_shots_on_target_count,
               s.away_possession
        FROM league_phase_stats s
        JOIN matches m ON m.match_id = s.match_id
        JOIN teams at ON at.team_id = m.away_team_id
    )
    SELECT team_name,
           SUM(goals) AS goals,
           SUM(shots) AS shots,
           SUM(shots_on_target) AS shots_on_target,
           ROUND(100.0 * SUM(goals) / NULLIF(SUM(shots), 0), 1) AS goal_conversion_pct,
           ROUND(100.0 * SUM(shots_on_target) / NULLIF(SUM(shots), 0), 1) AS shot_accuracy_pct,
           ROUND(AVG(possession), 1) AS avg_possession
    FROM team_stats
    GROUP BY team_name
    ORDER BY goal_conversion_pct DESC, goals DESC
    ''',
    conn
)
efficiency.head(12)


## 5. Interpretation

This project is descriptive. A relationship between possession, shooting, and results should not be interpreted as causal evidence.

The detailed dataset is limited to the league phase, so knockout-stage performance analysis should use the complete score/results table unless another validated detailed-stat source is added.


In [ ]:
conn.close()